Traffic sign detection


### 1. Mount Google Drive
We need to access your zip file stored on Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2. Extract Dataset
Replace the path below with the actual path to your zip file in Drive.

In [ ]:
import zipfile
import os

# Update this path to match your file location
zip_path = '/content/drive/MyDrive/Colab_Dataset/unified_dataset.zip'
extract_path = '/content/dataset'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Dataset extracted to {extract_path}")

Dataset extracted to /content/dataset


In [ ]:
import yaml

# Define the correct absolute paths
data_config = {
    'path': '/content/dataset/unified_dataset',
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['traffic_sign']
}

# Overwrite the data.yaml with verified paths
config_path = '/content/dataset/unified_dataset/data.yaml'
with open(config_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"Updated {config_path} with correct paths.")

Updated /content/dataset/unified_dataset/data.yaml with correct paths.


### 4. Optimized Training for Speed
Increasing the batch size and using a more efficient data loader can significantly reduce training time.

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.6 MB/s eta 0:00:00


In [11]:
from ultralytics import YOLO

# Load the model
model = YOLO('yolo26n.pt')

# Optimized Training Parameters
results = model.train(
    data='/content/dataset/unified_dataset/data.yaml',
    epochs=100,
    imgsz=640,       # Higher imgsz uses more VRAM but can improve accuracy
    batch=-1,        # Use batch=-1 for AutoBatch (YOLO will find the max batch size for your VRAM)
    device=0,
    single_cls=True,
    workers=8,       # Maximize CPU utilization for data loading
    exist_ok=True,   # Overwrite existing experiment folders
    name='traffic_sign_optimized'
)

Ultralytics 8.4.43 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/unified_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=traffic_sign_optimized, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overl

### 5. Download Model for Local Use
Run the cell below to download the `best.pt` weights to your local computer.

In [13]:
from google.colab import files

# Path to the best weights from the training run
model_path = '/content/runs/detect/traffic_sign_optimized/weights/best.pt'

if os.path.exists(model_path):
    files.download(model_path)
else:
    print("Model file not found. Please check the training output for the correct path.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 6. Local Inference Instructions

Once you have `best.pt` on your computer:

1. **Install requirements**:
   ```bash
   pip install ultralytics opencv-python
   ```

2. **Run Inference Script**:
   Create a file named `detect.py` and use the following code:

In [ ]:
from ultralytics import YOLO
import cv2

# 1. Load the model you downloaded
model = YOLO('best.pt')

# 2. Run detection on an image or video
# results = model.predict(source='your_image.jpg', conf=0.25, save=True)

# 3. For Real-time Webcam detection:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if success:
        results = model(frame)
        annotated_frame = results[0].plot()
        cv2.imshow("YOLO Detection", annotated_frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        break

cap.release()
cv2.destroyAllWindows()

### Tips for Faster Training:
1. **AutoBatch**: Setting `batch=-1` automatically determines the largest batch size that fits in your GPU memory.
2. **Mixed Precision**: YOLOv8/v11 (Ultralytics) uses `amp=True` (Automatic Mixed Precision) by default, which speeds up training on modern GPUs.
3. **Cache**: If you have enough system RAM, add `cache=True` to the train arguments to store images in RAM and avoid disk I/O bottlenecks.
4. **Monitor GPU**: Use the command below to see real-time VRAM usage.

In [12]:
!nvidia-smi

Wed Apr 29 08:17:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             31W /   70W |     509MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----